# Training residual-cue review and MASKED v2

This follow-up reuses the v1 audit and reviews five additional complete training
opinions. Seed 20261016 selected five of the 26 flagged training cases not fully
read in the previous review. The prior ten full training reads are preserved.
This is a targeted-population sample, not a corpus-wide masking error estimate.

Four missed direct-result sentences were found in three sampled opinions. Two
were absent from v1's candidate flags. General rules now cover explicit court
agreement about operative relief, “denial ... was proper,” and case-specific
“required the denial” forms. Facts, procedural history, cited cases and indirect
reasoning remain. `masked_training_review_v2.json` contains exact span decisions
and the complete training-only audit. Twenty-one cases in the new sampling pool
still lack full semantic review; no human review is implied.

`masker_v2` was frozen after training review and regression checks, before v2
processing of development, test and pending cases. No gold labels or annotation
quotations were used. Version 1 remains unchanged; v2 supersedes it for future
MASKED experiments. Any experiments using v1 need rerunning for a v2 comparison.


In [1]:
from pathlib import Path
import sys, json
sys.dont_write_bytecode = True
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "annotation/source/cases.jsonl").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run inside legal-judgment.")
MVP = ROOT / "annotation/submissions/becca/MVP"
sys.path.insert(0, str(MVP / "preprocessing"))
import masked_text_v2 as masker
import verify_masked_v2 as verifier
review = json.loads((MVP / "preprocessing/masked_training_review_v2.json").read_text())
print("New full-opinion training sample:", review["new_fully_read_training_case_ids"])
print("Confirmed missed sentences:", review["confirmed_missed_sentences"])
print("Regression checks passed:", verifier.check_rules())


New full-opinion training sample: ['NY-4306620', 'NY-4803674', 'NY-4807631', 'NY-5959503', 'NY-6042890']
Confirmed missed sentences: 4
Regression checks passed: 33


## Generate MASKED v2 with the frozen rules

Model exports contain only case ID and text. Every removed sentence is saved
with its exact FULL-relative offsets, reason and word count. Each case has FULL
and MASKED word counts, number/fraction removed, input/output hashes and flags.
All counts use whitespace-separated words, not encoder tokens.

Empty text is retained as an explicit flagged record and must not enter model
fitting until resolved in a new version. Fewer than 50 words and removal above
50% are review flags; they do not silently change the split or labels.


In [2]:
summary, audits = masker.build(ROOT)
print(json.dumps(summary, indent=2))
print("MASKED word range:", min(a["masked_words"] for a in audits),
      "to", max(a["masked_words"] for a in audits))


{
  "version": "masked_v2",
  "case_count": 114,
  "partition_counts": {
    "test": 16,
    "train": 81,
    "pending_or_excluded": 1,
    "development": 16
  },
  "full_words": 63619,
  "masked_words": 58254,
  "removed_sentences": 228,
  "removal_reasons": {
    "disposition_block": 92,
    "current_appellate_result": 24,
    "reviewed_ruling_result": 68,
    "operative_relief_result": 11,
    "disposition_synopsis": 23,
    "concluding_relief_result": 2,
    "express_relief_resolution": 2,
    "agreement_on_explicit_relief": 2,
    "response_to_explicit_relief_argument": 1,
    "nominal_reviewed_ruling_result": 2,
    "required_operative_relief": 1
  },
  "flag_counts": {
    "retained_cue_candidates_review": 43,
    "unbalanced_brackets_review_segmentation": 4,
    "over_half_words_removed_review": 1
  },
  "empty_case_ids": [],
  "near_empty_case_ids": [],
  "review_status": "automated masking with training sample assistant review; not human-reviewed",
  "labels_or_annotation_evi

## Verify and summarize review flags

Checks cover unique IDs, reconstruction from exact removal spans, source/output
hashes, word accounting, matching JSONL/text exports, and absence of annotation
fields from model inputs. Regression examples include both result directions,
party requests, prior cases, hypotheses, citations and sentence boundaries.
Only aggregate evaluation diagnostics are displayed. Evaluation candidates are
not manually corrected and do not inform rule changes.


In [3]:
print(json.dumps(verifier.check_exports(ROOT), indent=2))
for partition in ("train", "development", "test", "pending_or_excluded"):
    group = [a for a in audits if a["partition"] == partition]
    print(partition, "cases", len(group), "cue candidate cases",
          sum(bool(a["residual_cue_candidates"]) for a in group),
          "empty", sum(a["masked_words"] == 0 for a in group),
          "near-empty", sum(0 < a["masked_words"] < 50 for a in group))
print("Cases losing more than half their words:",
      [a["case_id"] for a in audits if a["removed_word_fraction"] > .5])
print("Frozen evaluation processing: no case-specific overrides.")


{
  "verified_cases": 114,
  "verified_output_files": 117
}
train cases 81 cue candidate cases 32 empty 0 near-empty 0
development cases 16 cue candidate cases 8 empty 0 near-empty 0
test cases 16 cue candidate cases 2 empty 0 near-empty 0
pending_or_excluded cases 1 cue candidate cases 1 empty 0 near-empty 0
Cases losing more than half their words: ['NY-5936535']
Frozen evaluation processing: no case-specific overrides.


## What the review establishes

The four confirmed training misses are handled by general rules. Retained cue
flags still include standards, history and indirect reasoning; they are not
confirmed leaks. Unreviewed training cases and all evaluation cases retain their
review limitations. No assertion of exhaustive cue removal or human adjudication
is made. The pending Sodhi case remains unassigned and labels remain provisional.
